# Realtime DQN Traffic Stage Viewer

This notebook launches a manual traffic viewer instead of a free-running animation. The first screen starts with visible queues on all four approaches. Use **Step Forward** to apply the model-selected phase or move active cars one small visual step. Use **Step Back** to rewind one saved visual step.

The DQN input is still the same 10-value vector used by `TrafficEnv`: aggregate queues for A/B/C/D, wait times for A/B/C/D, current phase, and time in phase. The viewer assumes right-hand traffic with exactly three inbound lanes per approach: left, straight, and right. It then sums those lane queues before sending the state to the model.

Car colors identify the approach they came from: A orange, B purple, C green, and D blue. The small label inside each car identifies its intended movement: L, S, or R.


In [1]:
from pathlib import Path
import sys

import torch

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "code").exists():
    project_root = project_root.parent

if not (project_root / "code").exists():
    raise RuntimeError("Could not locate the project root containing the 'code' directory.")

source_root = str(project_root / "code")
simulation_root = str(project_root / "code" / "Simulation")
for path in [source_root, simulation_root]:
    if path not in sys.path:
        sys.path.insert(0, path)

from State.traffic_env import TrafficEnv
from Neural_Networks.DQN_Implementation.dqn import DQN
from traffic_stage_viewer import run_stage_viewer


In [2]:
env = TrafficEnv()
state_dim = len(env._get_state())
action_dim = len(env.actions)
PHASE_NAMES = env.phases

model = DQN(state_dim, action_dim)
model_path = project_root / "code" / "Neural_Networks" / "DQN_Implementation" / "traffic_dqn_model1000.pth"
checkpoint = torch.load(model_path, map_location=torch.device("cpu"))
if "model_state_dict" not in checkpoint:
    raise RuntimeError("This is a legacy checkpoint without environment metadata. Rerun dqn.ipynb to retrain and resave the model.")
if checkpoint["state_dim"] != state_dim or checkpoint["action_dim"] != action_dim or checkpoint["phases"] != PHASE_NAMES:
    raise RuntimeError(
        "Checkpoint metadata does not match the current environment. "
        "Rerun dqn.ipynb to retrain and resave the model."
    )
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded DQN with state_dim={state_dim}, action_dim={action_dim}")
print("Phases:", PHASE_NAMES)
print("Action examples:", env.actions[:3], "...", env.actions[-1])


Loaded DQN with state_dim=27, action_dim=45
Phases: ['all_right', 'AC_forward', 'BD_forward', 'AC_left', 'BD_left']
Action examples: [{'phase_index': 0, 'duration_seconds': 10, 'label': 'all_right__10s'}, {'phase_index': 0, 'duration_seconds': 20, 'label': 'all_right__20s'}, {'phase_index': 0, 'duration_seconds': 30, 'label': 'all_right__30s'}] ... {'phase_index': 4, 'duration_seconds': 90, 'label': 'BD_left__90s'}


In [3]:
run_stage_viewer(model, PHASE_NAMES, seed=19)
